# Automated FinTech Portfolio Analysis Project

This project fetches stock market data using Python and prepares data for portfolio analysis and dashboard visualization.

!pip install yfinance  #yfinance is a finance data specific python library that will fetch data from yahoo finance site

In [1]:
import yfinance as yf 
import pandas as pd

In [2]:
stock_dictionary = {
    "Finance": ["JPM", "MS", "GS", "BAC", "C"],
    "Technology": ["AAPL", "MSFT", "AMZN", "GOOG", "NVDA"],
    "Healthcare": ["PFE", "JNJ", "MRK", "UNH", "ABBV"],
    "Automobile": ["TSLA", "F", "GM", "TM", "RIVN","TATAMOTORS.NS"]
}
tickers = []

for stocks in stock_dictionary.values():
    tickers.extend(stocks)

try:
    portfolio_data = yf.download(tickers, period="6mo")

except Exception as e:
    print("Download failed")
    print(e)

[**********************86%****************       ]  18 of 21 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  21 of 21 completed

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")


In [3]:
portfolio_data.shape

(124, 106)

In [4]:
portfolio_data.head(5)

Price          Adj Close       Close                                     \
Ticker     TATAMOTORS.NS        AAPL        ABBV        AMZN        BAC   
Date                                                                      
2025-12-15           NaN  273.601654  223.778107  222.539993  54.734585   
2025-12-16           NaN  274.100739  220.059128  222.559998  54.220177   
2025-12-17           NaN  271.335876  220.688782  221.270004  53.962975   
2025-12-18           NaN  271.685242  219.222855  226.759995  53.676094   
2025-12-19           NaN  273.162506  223.158279  227.350006  54.675228   

Price                                                                 ...  \
Ticker               C          F         GM        GOOG          GS  ...   
Date                                                                  ...   
2025-12-15  111.686775  13.335653  81.609459  308.916321  881.049927  ...   
2025-12-16  110.181770  13.355193  81.390450  307.328400  870.710083  ...   
2025-12-17  110.359993  13.003483  80.146095  297.671021  863.955566  ...   
2025-12-18  111.716469  13.013252  80.803108  303.353607  867.887451  ...   
2025-12-19  113.726440  13.159799  81.977776  308.207245  884.902466  ...   

Price         Volume                                                       \
Ticker           MRK        MS      MSFT       NVDA        PFE       RIVN   
Date                                                                        
2025-12-15  16735900   5219000  23727700  164775600   60360000   64341000   
2025-12-16  16663600   6840500  20705600  148588100  111695500   46207200   
2025-12-17  13180600   5796100  24527200  222775500   61255300   32172700   
2025-12-18  14312800   7993400  28573500  176096000   47062500   67245400   
2025-12-19  44980200  10639400  70836100  324925900   87590000  106165100   

Price                                                  
Ticker     TATAMOTORS.NS      TM       TSLA       UNH  
Date                                                   
2025-12-15           NaN  412000  114542200   6447200  
2025-12-16           NaN  292700  107608100   6334000  
2025-12-17           NaN  223600  106490400   4685600  
2025-12-18           NaN  352000   95168400   6354900  
2025-12-19           NaN  533700  103305400  10553000  

[5 rows x 106 columns]

In [5]:
# data quality checks
failed_tickers = portfolio_data["Close"].isna().all()
failed_tickers = failed_tickers[failed_tickers]
failed_ticker_list = failed_tickers.index.tolist()
print("Failed tickers:")
print(failed_ticker_list)

Failed tickers:
['TATAMOTORS.NS']


In [6]:
portfolio_data = portfolio_data.drop(
    columns=failed_ticker_list,
    level=1
)

#### Clean dataframe presentation

In [7]:
latest_date = portfolio_data.index[-1]

In [8]:
clean_table = pd.DataFrame({
    "Close": portfolio_data["Close"].loc[latest_date],
    "High": portfolio_data["High"].loc[latest_date],
    "Low": portfolio_data["Low"].loc[latest_date],
    "Open": portfolio_data["Open"].loc[latest_date],
    "Volume": portfolio_data["Volume"].loc[latest_date]
})

In [9]:
clean_table.head()

,Close,High,Low,Open,Volume
Ticker,,,,,
AAPL,291.130005,297.140015,289.619995,296.029999,38742100
ABBV,227.729996,228.399994,224.320007,227.500000,4084500
AMZN,238.550003,243.360001,233.589996,243.210007,51186600
BAC,56.020000,56.189999,55.259998,55.360001,32976900
C,139.830002,141.119995,138.210007,139.750000,10358400


In [10]:
# need date column as well
clean_table = clean_table.reset_index()

In [11]:
clean_table.head(1)

,Ticker,Close,High,Low,Open,Volume
0,AAPL,291.130005,297.140015,289.619995,296.029999,38742100


In [12]:
# add date column
clean_table["Date"] = latest_date

In [13]:
# stock data of lastest date
clean_portfolio_data = clean_table[["Ticker","Date","Close","High","Low","Open","Volume"]]
clean_portfolio_data.head()

,Ticker,Date,Close,High,Low,Open,Volume
0,AAPL,2026-06-12,291.130005,297.140015,289.619995,296.029999,38742100
1,ABBV,2026-06-12,227.729996,228.399994,224.320007,227.500000,4084500
2,AMZN,2026-06-12,238.550003,243.360001,233.589996,243.210007,51186600
3,BAC,2026-06-12,56.020000,56.189999,55.259998,55.360001,32976900
4,C,2026-06-12,139.830002,141.119995,138.210007,139.750000,10358400


In [14]:
# map sectors of respective stocks 
ticker_sector = {}

for sector, stocks in stock_dictionary.items():
    for stock in stocks:
        ticker_sector[stock] = sector

In [15]:
clean_portfolio_data["Sector"] = clean_portfolio_data["Ticker"].map(ticker_sector)

C:\Users\Ashlesha\AppData\Local\Temp\ipykernel_29828\1232821456.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_portfolio_data["Sector"] = clean_portfolio_data["Ticker"].map(ticker_sector)


In [16]:
clean_portfolio_data = clean_portfolio_data[["Ticker", "Sector", "Date", "Close", "High","Low", "Open", "Volume"]]

clean_portfolio_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume
0,AAPL,Technology,2026-06-12,291.130005,297.140015,289.619995,296.029999,38742100
1,ABBV,Healthcare,2026-06-12,227.729996,228.399994,224.320007,227.500000,4084500
2,AMZN,Technology,2026-06-12,238.550003,243.360001,233.589996,243.210007,51186600
3,BAC,Finance,2026-06-12,56.020000,56.189999,55.259998,55.360001,32976900
4,C,Finance,2026-06-12,139.830002,141.119995,138.210007,139.750000,10358400


In [17]:
clean_portfolio_data.isnull().sum()

Ticker    0
Sector    0
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [18]:
clean_portfolio_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Ticker  20 non-null     object        
 1   Sector  20 non-null     object        
 2   Date    20 non-null     datetime64[ns]
 3   Close   20 non-null     float64       
 4   High    20 non-null     float64       
 5   Low     20 non-null     float64       
 6   Open    20 non-null     float64       
 7   Volume  20 non-null     int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(2)
memory usage: 1.4+ KB


In [19]:
clean_portfolio_data['Date'] = pd.to_datetime(clean_portfolio_data['Date'],format='%d-%m-%Y')

In [20]:
clean_portfolio_data.describe()

,Date,Close,High,Low,Open,Volume
count,20,20.000000,20.000000,20.000000,20.000000,2.000000e+01
mean,2026-06-12 00:00:00,249.699499,251.899399,245.743753,249.324250,2.612607e+07
min,2026-06-12 00:00:00,14.840000,15.000000,14.610000,14.750000,5.399000e+05
25%,2026-06-12 00:00:00,109.662502,111.957500,109.247501,111.555000,6.082400e+06
50%,2026-06-12 00:00:00,220.884995,223.009995,218.490005,221.275002,1.428200e+07
75%,2026-06-12 00:00:00,330.080002,332.168243,324.997490,327.252502,3.874642e+07
max,2026-06-12 00:00:00,1062.750000,1073.739990,1048.060059,1053.000000,1.120018e+08
std,NaN,230.912594,233.027764,227.177891,228.954479,2.752601e+07


In [21]:
# check duplicates
clean_portfolio_data.duplicated().sum()

0

In [22]:
# clean dataset for the latest date
portfolio_data_with_sectors_lastestDay = clean_portfolio_data.copy()

In [23]:
portfolio_data.head(2)

Price            Close                                                 \
Ticker            AAPL        ABBV        AMZN        BAC           C   
Date                                                                    
2025-12-15  273.601654  223.778107  222.539993  54.734585  111.686775   
2025-12-16  274.100739  220.059128  222.559998  54.220177  110.181770   

Price                                                                 ...  \
Ticker              F         GM        GOOG          GS         JNJ  ...   
Date                                                                  ...   
2025-12-15  13.335653  81.609459  308.916321  881.049927  211.819275  ...   
2025-12-16  13.355193  81.390450  307.328400  870.710083  207.002731  ...   

Price         Volume                                                     \
Ticker           JPM       MRK       MS      MSFT       NVDA        PFE   
Date                                                                      
2025-12-15  10864100  16735900  5219000  23727700  164775600   60360000   
2025-12-16   8331300  16663600  6840500  20705600  148588100  111695500   

Price                                             
Ticker          RIVN      TM       TSLA      UNH  
Date                                              
2025-12-15  64341000  412000  114542200  6447200  
2025-12-16  46207200  292700  107608100  6334000  

[2 rows x 100 columns]

In [24]:
historical_stock_data = portfolio_data.copy()

In [25]:
historical_stock_data = (historical_stock_data.stack(level=1).reset_index())

historical_stock_data.head()

Price,Date,Ticker,Close,High,Low,Open,Volume
0,2025-12-15,AAPL,273.601654,279.630462,272.334020,279.630462,50409100
1,2025-12-15,ABBV,223.778107,224.614391,219.950906,220.619936,5971900
2,2025-12-15,AMZN,222.539993,227.929993,221.500000,227.929993,47286100
3,2025-12-15,BAC,54.734585,55.466619,54.477381,54.813722,32881300
4,2025-12-15,C,111.686775,112.399663,111.112497,111.340223,10692100


In [26]:
historical_stock_data.columns

Index(['Date', 'Ticker', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')

In [27]:
# map sectors of respective stocks 
ticker_sector = {}

for sector, stocks in stock_dictionary.items():
    for stock in stocks:
        ticker_sector[stock] = sector

In [28]:
historical_stock_data["Sector"] = historical_stock_data["Ticker"].map(ticker_sector)

In [29]:
historical_stock_data = historical_stock_data[["Ticker", "Sector", "Date", "Close", "High","Low", "Open", "Volume"]]

In [30]:
historical_stock_data.head(2)

Price,Ticker,Sector,Date,Close,High,Low,Open,Volume
0,AAPL,Technology,2025-12-15,273.601654,279.630462,272.334020,279.630462,50409100
1,ABBV,Healthcare,2025-12-15,223.778107,224.614391,219.950906,220.619936,5971900


In [31]:
historical_stock_data.isnull().sum()

Price
Ticker    0
Sector    0
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [32]:
historical_stock_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2480 entries, 0 to 2479
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Ticker  2480 non-null   object        
 1   Sector  2480 non-null   object        
 2   Date    2480 non-null   datetime64[ns]
 3   Close   2480 non-null   float64       
 4   High    2480 non-null   float64       
 5   Low     2480 non-null   float64       
 6   Open    2480 non-null   float64       
 7   Volume  2480 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(2)
memory usage: 155.1+ KB


In [33]:
historical_stock_data['Date'] = pd.to_datetime(historical_stock_data['Date'],format='%d-%m-%Y')

In [34]:
historical_stock_data.describe()

Price,Date,Close,High,Low,Open,Volume
count,2480,2480.000000,2480.000000,2480.000000,2480.000000,2.480000e+03
mean,2026-03-15 15:29:01.935483904,233.174158,235.990090,230.212398,233.040892,3.146971e+07
min,2025-12-15 00:00:00,11.070457,11.307469,10.971701,11.218589,1.460000e+05
25%,2026-01-29 18:00:00,94.658003,96.358794,93.292367,94.543717,7.187975e+06
50%,2026-03-16 12:00:00,211.874634,213.945320,209.005005,211.037015,1.678675e+07
75%,2026-04-29 06:00:00,305.671967,308.637940,301.596564,305.087502,4.188370e+07
max,2026-06-12 00:00:00,1092.609985,1098.359985,1050.000000,1092.819946,3.608079e+08
std,NaN,199.007685,201.608760,196.173161,198.795073,4.031371e+07


In [35]:
historical_stock_data.duplicated().sum()

0

In [36]:
# cleanups
historical_stock_data.columns.name = None
historical_stock_data = historical_stock_data.sort_values(["Ticker", "Date"])

In [37]:
historical_stock_data["Daily Return %"] = round(historical_stock_data.groupby('Ticker')['Close'].pct_change()*100,2)

In [38]:
historical_stock_data.head(2)

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
0,AAPL,Technology,2025-12-15,273.601654,279.630462,272.334020,279.630462,50409100,NaN
20,AAPL,Technology,2025-12-16,274.100739,274.989103,271.285991,272.314080,37648600,0.18


In [39]:
historical_stock_data["Daily Return %"].describe()

count    2460.000000
mean        0.065878
std         2.144290
min       -19.610000
25%        -1.040000
50%         0.005000
75%         1.120000
max        26.640000
Name: Daily Return %, dtype: float64

In [40]:
cumulative_returns = historical_stock_data.groupby('Ticker').agg(
                                                                 First_close = ('Close','first'),
                                                                 Last_close = ('Close','last'))
cumulative_returns

,First_close,Last_close
Ticker,,
AAPL,273.601654,291.130005
ABBV,223.778107,227.729996
AMZN,222.539993,238.550003
BAC,54.734585,56.020000
C,111.686775,139.830002
F,13.335653,14.840000
GM,81.609459,81.500000
GOOG,308.916321,358.160004
GS,881.049927,1062.750000


In [41]:
cumulative_returns["Cumulative Return %"] = round((cumulative_returns['Last_close'] - cumulative_returns['First_close']) / cumulative_returns['First_close'] *100 , 2)

cumulative_returns = cumulative_returns.sort_values("Cumulative Return %",ascending=False)
cumulative_returns.head()

,First_close,Last_close,Cumulative Return %
Ticker,,,
C,111.686775,139.830002,25.20
MS,175.870148,214.039993,21.70
UNH,338.468658,408.519989,20.70
GS,881.049927,1062.750000,20.62
MRK,99.522858,119.050003,19.62


In [42]:
# average trading volume
cumulative_returns["Total Volume"] = (historical_stock_data.groupby("Ticker")["Volume"].sum())

# sector mapping
sector_map = (historical_stock_data.groupby("Ticker")["Sector"].first())
cumulative_returns["Sector"] = sector_map

In [43]:
stock_performance_summary = (cumulative_returns.reset_index())

stock_performance_summary = stock_performance_summary[["Ticker","Sector","First_close","Last_close","Cumulative Return %","Total Volume"]]
stock_performance_summary.head()

,Ticker,Sector,First_close,Last_close,Cumulative Return %,Total Volume
0,C,Finance,111.686775,139.830002,25.20,1695769100
1,MS,Finance,175.870148,214.039993,21.70,812083300
2,UNH,Healthcare,338.468658,408.519989,20.70,1064348200
3,GS,Finance,881.049927,1062.750000,20.62,280873600
4,MRK,Healthcare,99.522858,119.050003,19.62,1396409500


In [44]:
# investor's porfolios 

investor_profiles = { 
    "Investor_A": {"capital": 10000, "weights": {"JPM": 0.30, "BAC": 0.30, "ABBV": 0.20, "JNJ": 0.20}},
    "Investor_B": { "capital": 10000,"weights": {"NVDA": 0.30, "TSLA": 0.25, "AMZN": 0.25, "AAPL": 0.20}}
}
investor_profiles

{'Investor_A': {'capital': 10000,
  'weights': {'JPM': 0.3, 'BAC': 0.3, 'ABBV': 0.2, 'JNJ': 0.2}},
 'Investor_B': {'capital': 10000,
  'weights': {'NVDA': 0.3, 'TSLA': 0.25, 'AMZN': 0.25, 'AAPL': 0.2}}}

In [45]:
for investor, details in investor_profiles.items():

    total_weight = sum(details["weights"].values())

    if total_weight != 1:
        raise ValueError(
            f"{investor} weights do not sum to 1. Current total: {total_weight}"
        )

-  validated that portfolio allocations summed to 100%, if not, the pipeline stops and raises an error to prevent incorrect return calculations

#### now lets calculate weighted return calculation for each investor

##### Investor A portfolio summary

In [46]:
investor_A_stocks = investor_profiles['Investor_A']['weights'].keys()
investor_A_stocks

dict_keys(['JPM', 'BAC', 'ABBV', 'JNJ'])

In [47]:
# filter the complete stock historical data to just gets stocks data held by the investor
investor_A_data = historical_stock_data[historical_stock_data['Ticker'].isin(investor_A_stocks)].copy()
investor_A_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
1,ABBV,Healthcare,2025-12-15,223.778107,224.614391,219.950906,220.619936,5971900,NaN
21,ABBV,Healthcare,2025-12-16,220.059128,223.177950,218.327546,222.754899,6405900,-1.66
41,ABBV,Healthcare,2025-12-17,220.688782,222.754886,219.262193,220.265731,5243400,0.29
61,ABBV,Healthcare,2025-12-18,219.222855,222.607307,219.193325,220.383798,4760400,-0.66
81,ABBV,Healthcare,2025-12-19,223.158279,225.765492,218.760441,219.350746,18968800,1.80


In [48]:
investor_A_data['Weight'] = investor_A_data['Ticker'].map(investor_profiles['Investor_A']['weights'])

In [49]:
investor_A_data.head(2)

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %,Weight
1,ABBV,Healthcare,2025-12-15,223.778107,224.614391,219.950906,220.619936,5971900,NaN,0.2
21,ABBV,Healthcare,2025-12-16,220.059128,223.177950,218.327546,222.754899,6405900,-1.66,0.2


In [50]:
investor_A_data["Weighted Return"] = (investor_A_data["Daily Return %"] * investor_A_data["Weight"])

In [51]:
portfolio_A_daily_returns = (investor_A_data.groupby("Date")["Weighted Return"].sum().reset_index())
portfolio_A_daily_returns.rename(columns={"Weighted Return": "Portfolio Daily Return %"},inplace = True )

In [52]:
portfolio_A_daily_returns.head()

,Date,Portfolio Daily Return %
0,2025-12-15,0.000
1,2025-12-16,-1.488
2,2025-12-17,-0.039
3,2025-12-18,-0.672
4,2025-12-19,1.137


In [53]:
initial_capital = investor_profiles["Investor_A"]["capital"]

portfolio_A_daily_returns["Portfolio Value"] = round((initial_capital * 
                                                (1 + portfolio_A_daily_returns["Portfolio Daily Return %"] / 100).cumprod()),3)
portfolio_A_daily_returns.head()

,Date,Portfolio Daily Return %,Portfolio Value
0,2025-12-15,0.000,10000.000
1,2025-12-16,-1.488,9851.200
2,2025-12-17,-0.039,9847.358
3,2025-12-18,-0.672,9781.184
4,2025-12-19,1.137,9892.396


In [54]:
portfolio_A_daily_returns["Portfolio Cumulative Return %"] = round(((portfolio_A_daily_returns["Portfolio Value"] - initial_capital) / initial_capital )*100,3)
portfolio_A_daily_returns.head(3)

,Date,Portfolio Daily Return %,Portfolio Value,Portfolio Cumulative Return %
0,2025-12-15,0.000,10000.000,0.000
1,2025-12-16,-1.488,9851.200,-1.488
2,2025-12-17,-0.039,9847.358,-1.526


##### Investor B portfolio summary

In [55]:
investor_B_stocks = investor_profiles['Investor_B']['weights'].keys()
investor_B_stocks

dict_keys(['NVDA', 'TSLA', 'AMZN', 'AAPL'])

In [56]:
# filter the complete stock historical data to just gets stocks data held by the investor
investor_B_data = historical_stock_data[historical_stock_data['Ticker'].isin(investor_B_stocks)].copy()
investor_B_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
0,AAPL,Technology,2025-12-15,273.601654,279.630462,272.334020,279.630462,50409100,NaN
20,AAPL,Technology,2025-12-16,274.100739,274.989103,271.285991,272.314080,37648600,0.18
40,AAPL,Technology,2025-12-17,271.335876,275.647872,271.136266,274.500011,50138700,-1.01
60,AAPL,Technology,2025-12-18,271.685242,273.122574,266.454969,273.102591,51630700,0.13
80,AAPL,Technology,2025-12-19,273.162506,274.090774,269.399478,271.645305,144632000,0.54


In [57]:
investor_B_data['Weight'] = investor_B_data['Ticker'].map(investor_profiles['Investor_B']['weights'])

In [58]:
investor_B_data.head(2)

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %,Weight
0,AAPL,Technology,2025-12-15,273.601654,279.630462,272.334020,279.630462,50409100,NaN,0.2
20,AAPL,Technology,2025-12-16,274.100739,274.989103,271.285991,272.314080,37648600,0.18,0.2


In [59]:
investor_B_data["Weighted Return"] = (investor_B_data["Daily Return %"] * investor_B_data["Weight"])

In [60]:
portfolio_B_daily_returns = (investor_B_data.groupby("Date")["Weighted Return"].sum().reset_index())
portfolio_B_daily_returns.rename(columns={"Weighted Return": "Portfolio Daily Return %"},inplace = True )

In [61]:
portfolio_B_daily_returns.head()

,Date,Portfolio Daily Return %
0,2025-12-15,0.0000
1,2025-12-16,1.0490
2,2025-12-17,-2.6450
3,2025-12-18,2.0695
4,2025-12-19,1.2395


In [62]:
initial_capital = investor_profiles["Investor_B"]["capital"]

portfolio_B_daily_returns["Portfolio Value"] = round((initial_capital * 
                                                (1 + portfolio_B_daily_returns["Portfolio Daily Return %"] / 100).cumprod()),3)
portfolio_B_daily_returns.head()

,Date,Portfolio Daily Return %,Portfolio Value
0,2025-12-15,0.0000,10000.000
1,2025-12-16,1.0490,10104.900
2,2025-12-17,-2.6450,9837.625
3,2025-12-18,2.0695,10041.215
4,2025-12-19,1.2395,10165.676


In [63]:
portfolio_B_daily_returns["Portfolio Cumulative Return %"] = round(((portfolio_B_daily_returns["Portfolio Value"] - initial_capital) / initial_capital )*100,3)
portfolio_B_daily_returns.head(3)

,Date,Portfolio Daily Return %,Portfolio Value,Portfolio Cumulative Return %
0,2025-12-15,0.000,10000.000,0.000
1,2025-12-16,1.049,10104.900,1.049
2,2025-12-17,-2.645,9837.625,-1.624


In [64]:
portfolio_A_daily_returns.iloc[-1]

Date                             2026-06-12 00:00:00
Portfolio Daily Return %                       1.639
Portfolio Value                            10484.806
Portfolio Cumulative Return %                  4.848
Name: 123, dtype: object

In [65]:
portfolio_B_daily_returns.iloc[-1]

Date                             2026-06-12 00:00:00
Portfolio Daily Return %                     -0.1085
Portfolio Value                            10517.085
Portfolio Cumulative Return %                  5.171
Name: 123, dtype: object

In [66]:
return_A = portfolio_A_daily_returns["Portfolio Cumulative Return %"].iloc[-1]
return_B = portfolio_B_daily_returns["Portfolio Cumulative Return %"].iloc[-1]

In [67]:
# expected annual return is

annual_return_A = round((portfolio_A_daily_returns["Portfolio Daily Return %"].mean()* 252),2)
annual_return_B = round((portfolio_B_daily_returns["Portfolio Daily Return %"].mean()* 252),2)
print(annual_return_A)
print(annual_return_B)

10.72
13.18


#### calculate risk to check which portfolio is more volatile

In [68]:
portfolio_A_daily_returns["Portfolio Daily Return %"].std()

0.9404481141051279

In [69]:
portfolio_B_daily_returns["Portfolio Daily Return %"].std()

1.528489486630371

In [70]:
annual_risk_A = (portfolio_A_daily_returns["Portfolio Daily Return %"].std() * (252 ** 0.5))

annual_risk_B = (portfolio_B_daily_returns["Portfolio Daily Return %"].std() * (252 ** 0.5))

print(round(annual_risk_A,2))
print(round(annual_risk_B,2))

14.93
24.26


In [71]:
# assuming risk-free rate as 0 
sharpe_A = round(annual_return_A / annual_risk_A,2)
sharpe_B = round(annual_return_B / annual_risk_B,2)
print(sharpe_A)
print(sharpe_B)

0.72
0.54


In [72]:
# creating the overall summary
portfolio_summary = pd.DataFrame({
                                "Portfolio": ["Conservative", "Growth"],
                                "Capital": [10000, 10000],
                                "Current Value": [
                                        round(portfolio_A_daily_returns["Portfolio Value"].iloc[-1], 2),
                                        round(portfolio_B_daily_returns["Portfolio Value"].iloc[-1], 2)],
                                "Total Return %": [
                                        round(portfolio_A_daily_returns["Portfolio Cumulative Return %"].iloc[-1], 2),
                                        round(portfolio_B_daily_returns["Portfolio Cumulative Return %"].iloc[-1], 2)],
                                "Annual Return %": [
                                        annual_return_A,
                                        annual_return_B],
                                "Annual Risk %": [
                                        round(annual_risk_A, 2),
                                        round(annual_risk_B, 2)],
                                "Sharpe Ratio": [
                                        sharpe_A,
                                        sharpe_B]
                                })

In [73]:
portfolio_summary.head()

,Portfolio,Capital,Current Value,Total Return %,Annual Return %,Annual Risk %,Sharpe Ratio
0,Conservative,10000,10484.81,4.85,10.72,14.93,0.72
1,Growth,10000,10517.08,5.17,13.18,24.26,0.54


In [74]:
portfolio_A_daily_returns["Portfolio"] = "Conservative"

In [75]:
portfolio_B_daily_returns["Portfolio"] = "Growth"

In [76]:
portfolio_daily_returns = pd.concat([portfolio_A_daily_returns, portfolio_B_daily_returns],ignore_index=True)

In [77]:
portfolio_daily_returns = portfolio_daily_returns[["Date","Portfolio","Portfolio Daily Return %","Portfolio Value",
                                                   "Portfolio Cumulative Return %"]]

In [78]:
portfolio_daily_returns = portfolio_daily_returns.sort_values(["Date", "Portfolio"])

In [79]:
portfolio_daily_returns = portfolio_daily_returns.reset_index(drop=True)

In [80]:
portfolio_daily_returns.head()

,Date,Portfolio,Portfolio Daily Return %,Portfolio Value,Portfolio Cumulative Return %
0,2025-12-15,Conservative,0.000,10000.000,0.000
1,2025-12-15,Growth,0.000,10000.000,0.000
2,2025-12-16,Conservative,-1.488,9851.200,-1.488
3,2025-12-16,Growth,1.049,10104.900,1.049
4,2025-12-17,Conservative,-0.039,9847.358,-1.526


In [81]:
historical_stock_data = historical_stock_data.sort_values(["Ticker", "Date"])

In [82]:
historical_stock_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
0,AAPL,Technology,2025-12-15,273.601654,279.630462,272.334020,279.630462,50409100,NaN
20,AAPL,Technology,2025-12-16,274.100739,274.989103,271.285991,272.314080,37648600,0.18
40,AAPL,Technology,2025-12-17,271.335876,275.647872,271.136266,274.500011,50138700,-1.01
60,AAPL,Technology,2025-12-18,271.685242,273.122574,266.454969,273.102591,51630700,0.13
80,AAPL,Technology,2025-12-19,273.162506,274.090774,269.399478,271.645305,144632000,0.54


In [83]:
print(historical_stock_data.shape)
print(portfolio_daily_returns.shape)
print(portfolio_summary.shape)

(2480, 9)
(248, 5)
(2, 7)


In [84]:
historical_stock_data = historical_stock_data.reset_index(drop=True)
portfolio_daily_returns = portfolio_daily_returns.reset_index(drop=True)
portfolio_summary = portfolio_summary.reset_index(drop=True)

In [85]:
historical_stock_data.to_csv("historical_stock_data.csv", index=False)
portfolio_daily_returns.to_csv("portfolio_daily_returns.csv", index=False)
portfolio_summary.to_csv("portfolio_summary.csv", index=False)
stock_performance_summary.to_csv("stock_performance_summary.csv",index=False)

### generate PDF report

In [2]:
import pandas as pd

In [3]:
portfolio_summary = pd.read_csv("portfolio_summary.csv")

portfolio_summary

,Portfolio,Capital,Current Value,Total Return %,Annual Return %,Annual Risk %,Sharpe Ratio
0,Conservative,10000,10484.81,4.85,10.72,14.93,0.72
1,Growth,10000,10517.08,5.17,13.18,24.26,0.54


In [4]:
max_return  = portfolio_summary["Total Return %"].max()
max_return

5.17

In [46]:
best_portfolio = portfolio_summary.loc[portfolio_summary["Total Return %"]==max_return,"Portfolio"].iloc[0]
worst_portfolio = portfolio_summary.loc[portfolio_summary["Total Return %"] == portfolio_summary["Total Return %"].min(),"Portfolio"].iloc[0]

In [47]:
print(best_portfolio)
print(worst_portfolio)

Growth
Conservative


In [41]:
return_difference = round(portfolio_summary["Total Return %"].max() - portfolio_summary["Total Return %"].min(),2) 

In [42]:
return_difference

0.32

In [48]:
print(f"Best Performing Portfolio: {best_portfolio}")
print(f"{best_portfolio} Portfolio Outperformed {worst_portfolio} Portfolio by {return_difference}%")

Best Performing Portfolio: Growth
Growth Portfolio Outperformed Conservative Portfolio by 0.32%


In [76]:
portfolio_daily_returns = pd.read_csv("portfolio_daily_returns.csv")

In [77]:
report_date = portfolio_daily_returns["Date"].max()
report_date

'2026-06-12'

In [88]:
report_text = f"""
📊 Portfolio Report

Report Date: {report_date}

"""

In [89]:
print(report_text)


📊 Portfolio Report

Report Date: 2026-06-12




In [90]:
for _, portfolio in portfolio_summary.iterrows():

    report_text += f"""

Portfolio: {portfolio["Portfolio"]}

Current Value: ₹{portfolio["Current Value"]}

Total Return: {portfolio["Total Return %"]}%

Annual Return: {portfolio["Annual Return %"]}%

Annual Risk: {portfolio["Annual Risk %"]}%

Sharpe Ratio: {portfolio["Sharpe Ratio"]}

"""

In [91]:
print(report_text)


📊 Portfolio Report

Report Date: 2026-06-12



Portfolio: Conservative

Current Value: ₹10484.81

Total Return: 4.85%

Annual Return: 10.72%

Annual Risk: 14.93%

Sharpe Ratio: 0.72



Portfolio: Growth

Current Value: ₹10517.08

Total Return: 5.17%

Annual Return: 13.18%

Annual Risk: 24.26%

Sharpe Ratio: 0.54




In [92]:
report_text += f"""

Portfolio Comparison

Best Performing Portfolio: {best_portfolio}

{best_portfolio} Portfolio Outperformed Conservative Portfolio by {return_difference}%

"""

In [93]:
print(report_text)


📊 Portfolio Report

Report Date: 2026-06-12



Portfolio: Conservative

Current Value: ₹10484.81

Total Return: 4.85%

Annual Return: 10.72%

Annual Risk: 14.93%

Sharpe Ratio: 0.72



Portfolio: Growth

Current Value: ₹10517.08

Total Return: 5.17%

Annual Return: 13.18%

Annual Risk: 24.26%

Sharpe Ratio: 0.54



Portfolio Comparison

Best Performing Portfolio: Growth

Growth Portfolio Outperformed Conservative Portfolio by 0.32%




!pip install reportlab

In [104]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

In [105]:
# creating a variable to save the path for report on my system
pdf_path = f"reports/portfolio_report_{report_date}.pdf"

In [106]:
# create an empty pdf document object 
doc = SimpleDocTemplate(pdf_path)

In [107]:
# get default text formatting styles
styles = getSampleStyleSheet()

In [116]:
elements = []

In [117]:
# add Title and recent date
elements.append(Paragraph("Portfolio Performance Report", styles["Title"]))

elements.append(Paragraph(f"Report Date: {report_date}", styles["Normal"]))

elements.append(Spacer(1, 12))

In [118]:
# add portfolio details of each
for _, portfolio in portfolio_summary.iterrows():

    elements.append(Paragraph(f"<b>Portfolio:</b> {portfolio['Portfolio']}",styles["Heading2"]))

    elements.append(Paragraph(f"Current Value: ₹{portfolio['Current Value']}",styles["Normal"]))

    elements.append(Paragraph(f"Total Return: {portfolio['Total Return %']}%",styles["Normal"]))

    elements.append(Paragraph(f"Annual Return: {portfolio['Annual Return %']}%",styles["Normal"]))

    elements.append(Paragraph(f"Annual Risk: {portfolio['Annual Risk %']}%",styles["Normal"]))

    elements.append(Paragraph(f"Sharpe Ratio: {portfolio['Sharpe Ratio']}",styles["Normal"]))

    elements.append(Spacer(1, 12))

In [119]:
# add the comparision section
elements.append(Paragraph("Portfolio Comparison", styles["Heading1"]))

elements.append(Paragraph(f"Best Performing Portfolio: {best_portfolio}",styles["Normal"]))

elements.append(Paragraph(f"{best_portfolio} Portfolio Outperformed Conservative Portfolio by {return_difference}%",styles["Normal"]))

In [120]:
print(len(elements))

20


In [121]:
doc.build(elements)

print("PDF Report Created Successfully!")

PDF Report Created Successfully!
